# 1) Import Configuration and Functions:

In [0]:
%run ../common/configuration


In [0]:
%run ../common/functions

# 2) Define Drivers Schema:

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType

drivers_schema = StructType([
    StructField("driver_id", StringType(), False),
    StructField("permanent_number", StringType(), True),
    StructField("code", StringType(), True),
    StructField("given_name", StringType(), True),
    StructField("family_name", StringType(), True),
    StructField("date_of_birth", DateType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True),
])

drivers_input_path = f"{processed_folder_path}/drivers/csv/drivers.csv"

drivers_df = spark.read \
    .option("header", True) \
    .schema(drivers_schema) \
    .csv(drivers_input_path)


drivers_dropped_df = drivers_df.drop("url")

# 3) Transform Drivers Data:

The steps included:

- Fill Null cells with "None".
- Create Surrogate Key.
- Add Data Source and File Date.

In [0]:
from pyspark.sql.functions import lit

drivers_with_audit_df = drivers_dropped_df \
    .withColumn("data_source", lit(v_data_source)) \
    .withColumn("file_date", lit(v_file_date))

drivers_date_df = add_ingestion_date(drivers_with_audit_df)
drivers_fill_df = drivers_date_df.fillna("None")


drivers_final_df = add_surrogate_key(
    drivers_fill_df,
    key_column_name="driver_sk",
    hash_columns=["driver_id", "permanent_number", "code", "given_name", "family_name", "date_of_birth", "nationality"],
)

 
print("Final columns going into the write:", drivers_final_df.columns)


# 4) Save the Processed Dataset to Delta Lake:

In [0]:
drivers_output_path = f"{processed_folder_path}/drivers/delta"
 
spark.sql("CREATE DATABASE IF NOT EXISTS f1_processed")
 
upsert_if_changed(
    input_df=drivers_final_df,
    db_name="f1_processed",
    table_name="drivers",
    output_path=drivers_output_path,
    merge_key_columns=["driver_id"],
)

In [0]:
display(spark.read.format("delta").load(drivers_output_path))

In [0]:
build_presentation_dimension(
    processed_location=f"{processed_folder_path}/drivers/delta",
    natural_key_column="driver_id",
    keep_columns=["driver_id", "permanent_number", "code", "given_name", "family_name", "date_of_birth", "nationality"],
    presentation_directory=f"{presentation_folder_path}/dim_drivers/delta",
    db_name="f1_presentation",
    table_name="dim_drivers",
)

In [0]:
display(spark.read.format("delta").load(f"{presentation_folder_path}/dim_drivers/delta"))

# 5) Save backup Drivers in CSV format:

In [0]:
drivers_backup_path = f"{presentation_folder_path}/dim_drivers/csv/dim_drivers.csv"

import io
import csv

backup_rows = [row.asDict() for row in drivers_final_df.collect()]
backup_fieldnames = drivers_final_df.columns

backup_buffer = io.StringIO()
backup_writer = csv.DictWriter(backup_buffer, fieldnames=backup_fieldnames)
backup_writer.writeheader()
backup_writer.writerows(backup_rows)

dbutils.fs.put(drivers_backup_path, backup_buffer.getvalue(), overwrite=True)
print(f"backup saved: {drivers_backup_path}")